# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matiasdibe23-sys/Flyrank-Starter-Notebooks/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Reason codes and tiers (validated in the capstone)

The queue uses the reason codes that were tested on held-back months in `capstone.ipynb` (Test A: May-June 2026, 19 clients seen in training; base decay rate 0.252). A code stays only if flagged pages decayed more than unflagged ones.

| Reason code | Rule | Decay rate, flagged vs not | Risk ratio | Suggested review |
|---|---|---|---|---|
| `IMPRESSIONS_FALLING` | impression momentum in the lowest quartile | 0.445 vs 0.185 | 2.41 | check demand and ranking visibility |
| `CLICKS_FALLING` | click momentum in the lowest quartile | 0.378 vs 0.189 | 2.00 | check freshness and search-result changes |
| `POSITION_SLIPPING` | position slip in the highest quartile | 0.330 vs 0.204 | 1.62 | review content against competing results |
| `LOW_CTR_FOR_POSITION` | CTR in the bottom quartile of its position band | 0.214 vs 0.264 | 0.81 | **dropped**: flagged pages decayed less |
| `HIGH_VOLUME` | clicks in days 1-15 at or above the 90th percentile | 0.238 vs 0.254 | 0.94 | **not a risk code**; kept only as a "senior review before editing" flag |

**Tiers** (by queue rank): `REVIEW_FIRST` = top 10%, `WATCHLIST` = next 20%, `NO_ACTION` = the rest. In the capstone, REVIEW_FIRST pages decayed at 0.39 (May) and 0.43 (June) against base rates of 0.24 and 0.27, so expect about four in ten flagged pages to be genuine decays. The NO_ACTION tier still held 35-38% of the clicks lost.

**Queue score here:** clicks in days 1-15 x how far click momentum is below 1 (the simpler "momentum risk x size" ranker). In the capstone it captured 28-30% of lost clicks in its top 10%, against about 40% for the Random Forest version.

**Caveat on thresholds:** the capstone took the quartile cutoffs from its training months. This notebook recomputes them on the June 2026 sample panel, so the flags are descriptive. No performance claim is made for this panel.

In [1]:
import duckdb
import numpy as np
import pandas as pd

PANEL_URL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet"
con = duckdb.connect()
if PANEL_URL.startswith("hf://"):
    con.execute("INSTALL httpfs; LOAD httpfs;")
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        con.execute(f"CREATE SECRET hf_auth (TYPE HTTP, BEARER_TOKEN '{hf_token}');")
    except Exception as e:
        print(f"Note: HF secret not created ({type(e).__name__}); trying unauthenticated read.")

H1, W1, W2S = "2026-06-15", "2026-06-07", "2026-06-08"
MIN_CLICKS, MIN_DAYS = 20, 7

raw_df = con.execute(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{H1}') AS impr_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '{H1}') AS clicks_h1,
        COUNT(*)             FILTER (WHERE report_date <= DATE '{H1}') AS days_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '{W1}')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{W1}'), 0) AS pos_w1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{W2S}' AND DATE '{H1}')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{W2S}' AND DATE '{H1}'), 0) AS pos_w2,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '{W1}') AS clicks_w1,
        COUNT(*)             FILTER (WHERE report_date <= DATE '{W1}') AS days_w1,
        SUM(gsc_clicks)      FILTER (WHERE report_date BETWEEN DATE '{W2S}' AND DATE '{H1}') AS clicks_w2,
        COUNT(*)             FILTER (WHERE report_date BETWEEN DATE '{W2S}' AND DATE '{H1}') AS days_w2,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{W1}') AS impr_w1,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '{W2S}' AND DATE '{H1}') AS impr_w2
    FROM read_parquet('{PANEL_URL}')
    WHERE month = '2026-06' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '{H1}') >= {MIN_CLICKS}
""").df()
raw_df = raw_df[raw_df.days_h1 >= MIN_DAYS].copy()

# Features from days 1-15 only (same definitions as the capstone)
rc1 = raw_df.clicks_w1.fillna(0) / raw_df.days_w1.replace(0, np.nan)
rc2 = raw_df.clicks_w2.fillna(0) / raw_df.days_w2.replace(0, np.nan)
raw_df["click_momentum"] = ((rc2 + 1) / (rc1 + 1)).fillna(1.0)
ri1 = raw_df.impr_w1.fillna(0) / raw_df.days_w1.replace(0, np.nan)
ri2 = raw_df.impr_w2.fillna(0) / raw_df.days_w2.replace(0, np.nan)
raw_df["impr_momentum"] = ((ri2 + 1) / (ri1 + 1)).fillna(1.0)
raw_df["pos_slip"] = (raw_df.pos_w2 - raw_df.pos_w1).fillna(0.0)

# Quartile cutoffs recomputed on this panel (see the caveat above)
q_click = raw_df.click_momentum.quantile(0.25)
q_impr = raw_df.impr_momentum.quantile(0.25)
q_pos = raw_df.pos_slip.quantile(0.75)
q_vol = raw_df.clicks_h1.quantile(0.90)
raw_df["CLICKS_FALLING"] = raw_df.click_momentum < q_click
raw_df["IMPRESSIONS_FALLING"] = raw_df.impr_momentum < q_impr
raw_df["POSITION_SLIPPING"] = raw_df.pos_slip > q_pos
raw_df["HIGH_VOLUME"] = raw_df.clicks_h1 >= q_vol
RISK_CODES = ["IMPRESSIONS_FALLING", "CLICKS_FALLING", "POSITION_SLIPPING"]
ALL_CODES = RISK_CODES + ["HIGH_VOLUME"]
raw_df["reason_codes"] = raw_df[ALL_CODES].apply(lambda r: ", ".join(c for c in ALL_CODES if r[c]) or "(none)", axis=1)

# Queue: clicks x how far click momentum is below 1, then tiers by rank
raw_df["queue_score"] = raw_df.clicks_h1 * (1.0 - raw_df.click_momentum).clip(lower=0)
rank = raw_df.queue_score.rank(ascending=False, method="first") / len(raw_df)
raw_df["tier"] = np.where(rank <= 0.10, "REVIEW_FIRST", np.where(rank <= 0.30, "WATCHLIST", "NO_ACTION"))
playbook_queue = raw_df.sort_values("queue_score", ascending=False).reset_index(drop=True)

print(f"Pages in the queue (>= {MIN_CLICKS} clicks and >= {MIN_DAYS} days with data in days 1-15): {len(playbook_queue):,}")
print("\n=== CONTENT ACTION QUEUE, TOP 10 (no ids shown) ===")
print(playbook_queue[["tier", "clicks_h1", "click_momentum", "reason_codes"]].head(10).round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in the queue (>= 20 clicks and >= 7 days with data in days 1-15): 4,547

=== CONTENT ACTION QUEUE, TOP 10 (no ids shown) ===
           tier  clicks_h1  click_momentum                                                         reason_codes
0  REVIEW_FIRST    97954.0           0.808                                                          HIGH_VOLUME
1  REVIEW_FIRST    88396.0           0.853                                                          HIGH_VOLUME
2  REVIEW_FIRST      483.0           0.044                     IMPRESSIONS_FALLING, CLICKS_FALLING, HIGH_VOLUME
3  REVIEW_FIRST      350.0           0.321  IMPRESSIONS_FALLING, CLICKS_FALLING, POSITION_SLIPPING, HIGH_VOLUME
4  REVIEW_FIRST      300.0           0.318  IMPRESSIONS_FALLING, CLICKS_FALLING, POSITION_SLIPPING, HIGH_VOLUME
5  REVIEW_FIRST      567.0           0.650                                          CLICKS_FALLING, HIGH_VOLUME
6  REVIEW_FIRST      479.0           0.631  IMPRESSIONS_FALLING, CLICKS_FALLING, POSI

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and known limits

* **Intended use:** internal decision-support for content and SEO teams, to decide which pages to open first for a manual review. The queue orders pages; a person decides what to do.
* **What was tested, and where:** the reason codes and tiers were validated in `capstone.ipynb` on May-June 2026 with 19 clients seen in training: REVIEW_FIRST pages decayed at 0.39-0.43 against base rates of 0.24-0.27. On 6 unseen clients the capstone could not separate the model from the momentum rule (the queue here uses the momentum rule).
* **Known limits:**
  * Nothing here shows that refreshing a page recovers clicks. The data are observational and have no recorded refresh events.
  * Click loss beyond the typical page is a proxy for "needs review"; seasonality, search-result changes and demand shifts also produce it.
  * 35-38% of the clicks lost sat in the NO_ACTION tier, so the queue is not a safety net.
  * This notebook runs on a one-month sample panel (June 2026). Cutoffs are recomputed on it and are not validated here.

In [2]:
# Section 2 check: how the queue is distributed (counts only)
tier_counts = playbook_queue.tier.value_counts().reindex(["REVIEW_FIRST", "WATCHLIST", "NO_ACTION"])
summary_use = tier_counts.rename("Count").to_frame()
summary_use["Share"] = (summary_use.Count / len(playbook_queue)).round(3)
print("=== QUEUE TIER DISTRIBUTION ===")
print(summary_use.to_string())

code_share = playbook_queue[ALL_CODES].mean().round(3).rename("Share of pages flagged")
print("\n=== REASON-CODE FLAG RATES (about 0.25 by construction for the quartile codes, 0.10 for HIGH_VOLUME) ===")
print(code_share.to_string())

=== QUEUE TIER DISTRIBUTION ===
              Count  Share
tier                      
REVIEW_FIRST    454    0.1
WATCHLIST       910    0.2
NO_ACTION      3183    0.7

=== REASON-CODE FLAG RATES (about 0.25 by construction for the quartile codes, 0.10 for HIGH_VOLUME) ===
IMPRESSIONS_FALLING    0.250
CLICKS_FALLING         0.250
POSITION_SLIPPING      0.250
HIGH_VOLUME            0.102


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and the no-go list

1. **Human-in-the-loop rules (suggested team policy, not validated):**
   * **Senior review before editing high-volume pages.** Pages flagged `HIGH_VOLUME` (clicks at or above the 90th percentile of the panel) get a senior check first. The reason is the cost of a mistake, not risk of decay: in the capstone, flagged pages decayed at 0.238 against 0.254 for the others.
   * **Check the size effect.** The score favors big pages. In the top 10 of the capstone's May queue, 4 pages decayed and four of the six that did not were flagged only `HIGH_VOLUME`. A reviewer should read the reason codes, and treat a page with no falling code as "big, not necessarily at risk".
   * **Intent check for `POSITION_SLIPPING`.** Verify search intent manually before changing titles or headers.

2. **The no-go list (do not automate; suggested policy):**
   * Core brand landing pages and homepage assets.
   * Legal terms, privacy policy and compliance documentation.
   * High-converting transactional product or sales pages.

In [3]:
# Section 3 check: how many REVIEW_FIRST pages are big pages with no falling or slipping code
review = playbook_queue[playbook_queue.tier == "REVIEW_FIRST"]
no_risk_code = ~review[RISK_CODES].any(axis=1)
print(f"REVIEW_FIRST pages: {len(review):,}")
print(f"  with HIGH_VOLUME: {int(review.HIGH_VOLUME.sum()):,}")
print(f"  with no risk code (size only, no falling or slipping signal): {int(no_risk_code.sum()):,}")
print(f"  with HIGH_VOLUME and no risk code (read these before acting): {int((review.HIGH_VOLUME & no_risk_code).sum()):,}")

REVIEW_FIRST pages: 454
  with HIGH_VOLUME: 218
  with no risk code (size only, no falling or slipping signal): 73
  with HIGH_VOLUME and no risk code (read these before acting): 68


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

1. **What to track each month:**
   * The market move: the median page ratio (second-half clicks/day over first-half). In 2026 it was 0.89-0.97 from January to May and 0.76 in June, a market-wide shift that the adjusted label removes.
   * The adjusted decay rate (it was 0.24-0.31 across January-June 2026).
   * Whether REVIEW_FIRST pages still decay more than the base rate (0.39-0.43 against 0.24-0.27 in May-June). If that gap closes, the queue has gone stale.
   * Each reason code's risk ratio on the latest month; drop a code whose ratio falls to 1 or below.
   * The share of flagged pages that were actually reviewed.

2. **Triggers to recalibrate (starting points taken from the six months seen, to be tuned):**
   * A month's median page ratio falls outside 0.85-1.0, or its adjusted decay rate leaves 0.20-0.35.
   * A reason code stops separating decayers.
   * Several months of data, with recorded refresh events, become available. Only then can the real business question (the gain from refreshing) be tested.

In [4]:
# Section 4 check: a baseline snapshot of the monitored quantities on this panel
print("=== MONITORING BASELINE (June 2026 sample panel) ===")
print(f"Median click momentum (days 8-15 vs 1-7):      {playbook_queue.click_momentum.median():.3f}")
print(f"Median impression momentum:                    {playbook_queue.impr_momentum.median():.3f}")
print(f"Median position slip (positive = sliding down): {playbook_queue.pos_slip.median():.3f}")
print(f"Share of pages flagged by a risk code:          {playbook_queue[RISK_CODES].any(axis=1).mean():.3f}")
print("Re-run each month and compare; the thresholds in the markdown above are starting points.")

=== MONITORING BASELINE (June 2026 sample panel) ===
Median click momentum (days 8-15 vs 1-7):      0.875
Median impression momentum:                    0.916
Median position slip (positive = sliding down): 0.174
Share of pages flagged by a risk code:          0.488
Re-run each month and compare; the thresholds in the markdown above are starting points.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the research paper

The queue is exported **without page or client ids** to `work/outputs/w07_content_action_queue.csv` (CSVs in `work/` are gitignored by the repo rules, so it stays local). The tier chart goes to `work/figures/w07_tier_distribution.png`.

In [5]:
import os
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = ["tier", "clicks_h1", "click_momentum", "impr_momentum", "pos_slip", "reason_codes", "queue_score"]
queue_export_path = "work/outputs/w07_content_action_queue.csv"
playbook_queue[export_cols].round(4).to_csv(queue_export_path, index=False)
print(f"[EXPORTS] Action queue (no ids) saved to {queue_export_path} ({len(playbook_queue):,} rows).")

plt.figure(figsize=(7, 3.6))
tier_counts.plot(kind="bar", color="#2563EB", rot=0)
plt.title("Action queue: pages per tier (June 2026 sample panel)")
plt.xlabel("Tier"); plt.ylabel("Number of pages")
plt.tight_layout()
figure_path = "work/figures/w07_tier_distribution.png"
plt.savefig(figure_path, dpi=200)
plt.close()
print(f"[EXPORTS] Figure saved to {figure_path}.")

[EXPORTS] Action queue (no ids) saved to work/outputs/w07_content_action_queue.csv (4,547 rows).
[EXPORTS] Figure saved to work/figures/w07_tier_distribution.png.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.